# AeroVision Lab — free Colab GPU pilot
1. Choose **Runtime → Change runtime type → GPU** (a free GPU option, if available).
2. Run the cells in order. Approve Google Drive access when prompted.
3. Keep this notebook private: it includes a snapshot of your private project's source.

Training runs on Google's GPU. Do not connect to a local runtime.
This notebook contains the complete pipeline, so no GitHub token is needed.
Checkpoints save to `My Drive/AeroVisionLab/runs/` every epoch. Colab resources
are not guaranteed; if a session ends, rerun the notebook to prepare the same data
and resume from the saved checkpoint. A failure during the current epoch can lose
that epoch. Select another saved epoch checkpoint if `last.pt` is damaged.

First milestone: YOLO11n, 800 training / 200 validation images, 640-pixel input,
batch 2, 20 epochs. Batch stays small to preserve the planned pilot setup.


In [ ]:
import os, sys, pathlib, subprocess, json, csv, zipfile, io, base64, hashlib
from google.colab import drive
import torch
assert torch.cuda.is_available(), 'Select a hosted GPU runtime, then rerun this cell.'
print('Hosted GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PERSIST = pathlib.Path('/content/drive/MyDrive/AeroVisionLab')
PERSIST.mkdir(parents=True, exist_ok=True)
PROJECT = pathlib.Path('/content/aerovision-lab')
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
os.environ['YOLO_CONFIG_DIR'] = str(PROJECT / '.state/ultralytics')


Source snapshot SHA-256: `d51f504e17ef3ea8ebf1afc9cd0dbb657a1fda1b994a246f3197d09ea0cf74ed`. The next cell installs this snapshot into the hosted runtime.

In [ ]:
bundle = base64.b64decode('UEsDBBQAAAAIAAAAIQB4tUodYAEAAFECAAAOAAAAcHlwcm9qZWN0LnRvbWxVUctqwzAQvOsrhM6xsB0IaakMPfXSQqD0UIwIsryO1ciSKslJ/fdFdsjjOLuzO7OzdTMq3WZhChEGjjz8jspDwAzXJEAcXbRWh4pttoSjhdsIeQTTYobvGHTu7QeIgiBUO29/QEaOjBggMQV4e1JBWZNp0RB0Ap9AauW0oDlBLQTplYuX6ucgtMZvuy8Mfw68GsDEgDvrsQCvhMYhETLbJB3cQgSZRsn1hMxNsV92VWxNiyJJODAtGKkuF446eqGnqGRgbEvXtCy3ZIWJGQc3Vayg5Wb1sk6VndLanitW5DOavl8/3iu2ScBN0kp7yamkOU1RXSOgdj5J6OxenaMWTrMFN0UIsWKzrh+7rmI5fXrYsOQSOLqF+BgplVo9D0KZFH0yQu8e44Q8igME2inTcnTuwaeP1CR4OcvMA4sNqozaL4YDR6niROyXrBIKt4FklSOtDGQazCH2mOGiyNE/UEsDBBQAAAAIAAAAIQD76a7BuwkAAEIUAAAJAAAAUkVBRE1FLm1kdVhtbxtHDv4uQP+BQD5ca0hr2U1ybdwrYDhtGjQvRhynOBTFhZqltHOeHW6Hs5LVX38gZ3elvtwXQ5Z2OHx5+PDhPoFrSvzJi+cIb3A9n81ndy2GsOT1f8llqCmTy/orPXaUfEsxC3CET15eJo70AkgyroOXBhASdYnr3vl1IFijUPCRFpAbiuDjjiT7LWaCRMKhV7sLyD5QDT5uKFF0BBhrqKkLfNDLKnXpY+MFvACqlZo6ijXFDL7tAulDaB5KxpSphk3iFsQlzK6p4B1DTuijj1u9tg9ZoMEdwZooQnGXajhQuerJE7jLuCW4eDFFQxoAZJQHfeKigk8YfK1x6PcUdz5xVD/M91e397D3WdMhDacM0vIDQepjNZ9dVnCbqMNEUwbLw/TYBe98BhdQBFrsOh+3C5Au+AwtRr8hyZqCGjBGHmJ2DbkHqeazryr4qGECwr/fv3l/cSFT/gEzPH+6gs4/UpAK7jR6q8COahDK2cetLGBMrDvAjpJCQhYgRPUCGkz1Xp3W6+3Ojn3M0KA0pNc/reD7HYZek8IRdiVB6mGNGa/0Oko7gkySlzXtYMMJNj5iACrn9FncZEogFBRzcVvSO+FOi5XZcajms2cV3HBreXz+dLWAb56vzLeLy69X4GPX5xOMyVgPxzEnDoq3CRNCue8quGmYhQBhrbAB8b9rdTHDxmvWQwBX7qu1kqUOiRwnLccBatp5i2HAK8HGJ8mwZQyK3Mx/gyYzcmwf7hPwPkJLKH0yYEsFb9UfdRQj54bSn1D/DwF6RJch9u2akrVJ5Awo0rdU/xHTly/0dOId6dfLKYX7hgMtfavPqE+8oxQKAv/Snr3ot46jeMlaFXGcin/1EcXSd10iMV4ZEVbpla9PWIBS4iQQCZNdA2vuY43JkyxGHwod9GpWj7SUtj5uzdQH6kp7nfBVIochLKCjtCyddH27gA0GIehYfPY7Nd5Sy+lgtgNmgzwGjlvxNdnNWvD2+tbu+UTJbw4aGxu/tFxTgC5R7Y0aCxgGGNMJd6kXG04tRkdWiO93lA5jCeyJtpcMazqCa2QO1N6cOpijgUawJWuokhV2vRqh+rQzrNyFCjjCJhHBK+ZtILjhUBhe4Rmwy9yBNH0WqBV2fawpGXsFxnoBckKcDQq0rC5lBoSGRW99dXtfFWscagjsMNj51MeoQBVIlH2i+gpu3rwu1s6HSCHyHmraYB+yGr0xW8auL3kf1QX45Tic/mPOV747xPWvX0TOtGZ+kPO/f+LLwrTvO4rQ5NzJi/NzZz8rD2FyTbW1nFSO2/OB1Kz/z85+UCB+B/eduTDedHZWOHagibOzD33MvtUnbxqMW+N3+yIfOv321e392ZlZ7o1WhpQNTxXG/NCXsjoKQUALlrQERvGdYWQs3cvkdwToHInAXqdpl7jt8tDgNh9HV4HaNdUCqCzuN145nvvkCCRiJw1nq21keOXzj/0aMj/oeNZq/dZruar57CeiDrxSrt9hpgpeYkYhnbm4JdFZe1B8WdLHYQuSOeGWrjRMwJT9Bl2ZWfPZcWqIoXvA2+e3hxLdSSnf4PpcSfb8MxC6BqhjneQfSIE1zIX5bIo30br3oZZji0i/Vl8LRysPmnTo48ZHL40mxNrr6FI1n73OgEHYfiqmCq8OuVv3sQ6kRTJvK/jBOuv2fqyKXVaqO58FvyH9JLDDdLiC/IcCaTcpRx9Ie59UW5HrFR4+wkHHQEkrOsf9JIEIOh84K55kmPFxAV+vVqWx4BwuV6vTyVtKtTiZ/Ythul3OZ+ru5aqkVip4nYt8kqKebt7fvF92icyyshL5baPTSN0gTMFTGhre1M18pm0dKGuva+7Kb8XhPSq4M6XUK2JhTRtONExPmxwwnR5qPdDYze29TlYK3CnRTdxlxk91V6IWvRLxDn1AFZ+qL4bSUW0jPNZmTxWIKMloCeYzRapWcU3RNS2mBwEuLDiRn5Gvl4EySw7UCtXQeIW8V3d+9rHmvcxnp35ZU2mYpbafa3ZyPrSZzfelxVLlx/z5aqh6omyx+CzzWZfIR8lomuX28JGTa060RwHqqNYAXe4xhINixC7UwDXCljLq3BgT+/YoMabZUSSBBqeab/n0cmwjQS2OAG823nkMA97Ui1O0HYV5OCzGKOYzMPGUKULGtKVB4CouJy0pVi2vRy2VRYioWDNN7RKLFCGsGgLA6K5mKl10FFBHFxwBb2DnpaRDfOsDJhBH0eTqEt4nvzX5qdNARYG5U6Sn5q7QnC0Kdz9eLy+fPR/Erqb1iPBRmBdts42sM9wklGq1UZWs+VEvSATc+jztKNrER5iV9FiP2bZjqYPgWz9ofd4UHK5RvDuecxwHALxQxw7gMKqkyIlQr0I94R62ScVVdZR9XjgWMukOjh1n5mCr3USjQxJMkrT4+JKy/Our1WqhV260XI7UyVW1Wl1Y0O/e3sFrvodV9c9qykaibQGn9byqwqJZUE5Umku8r0uejiXWje9U5dhQfPStpWNhxVdfJ1xOG9WgxThZuD//Sd0WRXvUbzahj+iLlPecHv6yRZQxBna8tK3FO6lb+Phnu1jXAtvAawwF9Uu0Jerd27sKrvMocUsOLxeAStVLo+qih7HeqXYUwDX3GZ5dXI5LnMb1ZlCuPrrQ11pHWzF0GtnqqCQ26fbFKJ2nqai6K2GUDaUFrPusbtDjYMqytdyoEzU5VpquymZrqRGQQ3RN4uh/HwBvk1qlMewxtVRD31kl7+i3XlOrvKHmjpvEBnfcJ4HA+1GQ61rRt12B+44SdIQPkJvE/bbp+tJmquivb5+tFn+U/te353+n/s25lmqP8fybZ7lZdpScOhRUBFsGrdN0DUranLEA/kTfL8b5ZxSwGHlR/yksOPCvefexIRV8yTbXwElheDglyuElxJUqsGkdVklgUkCluCpPqnUOKY8bZ3/QndNk06mOItsnBnn0UVdMZ1mz9yTHgZv6qISso4rGkTCf1T6RM+dGlfq5EPtyaSvGtwdsw3ewXEblgm/1r/5XBBV822FuzjOfB5Rcdfm7z9V89o72ZTtOZCVBlTZebJfXMXS8UteIHfsaxAebGFZuLcDJe5ppebU5PjC0486219f/90XQVWE1pYYik4aNzXp/ZE2KW12slIfns1/uQ04YDtk7+fWLcWPQYV31x19sWfiymGnKHiaUC5WbIv1lIqBM2B4NbX1u+rUdH5+YPiwHYf1lNYR5MpASzWcneBgvghNvdc+nLNB63aXtbdfoF6cJtopGx22ZPdV8dt+JTocWhDd5eq8zHsyUWlGyDVan3JBPWpJO38ooArntONrbifnsf1BLAwQUAAAACAAAACEATCns5zIAAAAwAAAAGgAAAHNyYy9hZXJvdmlzaW9uL19faW5pdF9fLnB5U1JS8sxLSS1IzUtJzStRSEwtykzMUUhJLUlNLsnMz1NIrShILcrMTc0rKdZTUlLiAgBQSwMEFAAAAAgAAAAhAKpxfQnLAgAAywgAABUAAABzcmMvYWVyb3Zpc2lvbi9jbGkucHmdVlFr3DAMfs+vEH7ylWtaShljkIeysVIo7TG2wSgl+BLlzltie7aSbvv1w46Tdq3v2vUpkaPv8ydZltJY3UFZNj31FssSZGe0JRBKaRIktXJZNq3ZjRHW4WR/d1pN79pljWcygratXE80K0HbLMtqbKATUvHFuwwAQLsc1SCtVrlDqrERfUucfbu+vC7fX199vDgvP1x8YktwZLnnyKu7mi/gCFjuSBAy/9q3ZEX7m2Tl2GIRiIM+C8WsNT+zm75DRavwhdfoKiuND6xgZ2j1V+mkVnAp1uA60bZwvvoCRhpspUI2srp+DUXkzkVdl65fj5bzhFSwSnedUDVbgsWfvbRYF59tjzM8oEYIZ7WuSFu2hC22pmArKxXBgNYLcSBUHTRI1WjbhSOIMoxFA8UTOr8s7KTVW+GziIFzdnhotSa2BPptsPDpXELMebA4qwWJOYUpArJCqolBKroneHt8vAc3iDaJOtmLcoh1EnZ6MqIaSYlEjCJnjye0Icp/spA4rBQQja62bkcgu2Gy27g/SdSb0z2wtaBqm95sN0iJDtPVl8wEDrLy/hMzq0y/J3MvOZAnoDttf6BNp21P+BZdH2KZfK+0wvmuCNoCaWiFo9xQlIxDohoq3T24Fji8rhwSuDuUmy2510BfUBEJFMkW/xukB7StMBOuabV4mP/8ZHde9hRHwr2VnaR9h5wS15Pp93Yk2yt3NB6hdL4DjlzCbhwUMAjreOzH4eHZHY9OsRmPU8DlRhs+N+jRQzb3TgVMDXmcTaEu/SjL8ZdBK73maZzFoeWXstnZ+P7N/TDM674zjj/w4oslSFWj8pc3pqN9tPvUvx9v7+tz2jj6PN40LnMf5w0LXf52GaK+id1wNn0nno1wn293CRqRz2cj+N1LCiY/OPBbTMwOn6eJF/WeKC7MVFkmGyhL3+LKMigsS/83UZZR5Phrkf0FUEsDBBQAAAAIAAAAIQCZMW3u7QkAAFgcAAAWAAAAc3JjL2Flcm92aXNpb24vZGF0YS5weZVY64/jthH/7r9iwKKIdKeVvds0uLrnAinuAhyQ5ILLA2i3C4OWRjbvJFIlqbUdw/97MXxI8mN3U33YlcnhvDnzGzHG3qmtrBUvQWmxFpLX8Jsw77SSCFwXG/GIBrgsoeFfEEq0qBshhbGiANPwugbTrQxak08mP6tOFziHjbWtmU+na2E33SovVDONPPuXm3fccoN28q0XAo3QWuk5/Fpbzeu9FYUBbhxn+FFB1xqrkTdQKPmI2ggloVAlgjCAOyw6i2U+YYxNJpVWDSyXVWc7jcsliKZV2gKXUlluhZJmMglrG242tVjFn5+NkvFdc1mqJv7qdF2LVa7xvx0aG1d/F20lavQSW26JVxT3E7ebXs6eN7Wn+unD95HiQ8PXOJn8+O0P73+GBdxPAABYiyUaqwWXLIsrqq0x/lqJYl8MPwuu4+vjcMTqrvgy/Dg9wrdSyPXN+fKqM/G1UVYR24fJDx8+ffr4CRbArgS1G0I19aGaaqyRGzTTMqTV9HGWz/IZm0wmJVZQijUam5Cv5s5FKdz8A4zVcyd5A4sYk9xs+N1fv0lSt7EVduM8nKsWZcL0iqXADVT+HD2V0rCqVfEFhARhUSc1b1Yln0OVa+Rlcju7+xpeAf1LM1gxlg6HnfC8a0tuMXFcvFyNttMSNvkGd0H3NJjScm1wqdXWJBZ3dk5WZLAVJZkmpM1gg2K9se6HM9N2bY33tTA2A/r74OUzxj55MY+8FiVYrtdo/a0Ta6k0lqBxTYlL/O0GVmqHBopatC2WYBXYDYKgdMrpBhDTwCTrOSzg/iGD+4dJ9FUtJJKrSPnctLWwtGKSkVdEBVJZR5lTTrbjTXoKJa2QHfaLWm1JUlUrbpPHeCh1Ah9JmudF0hKWsZRE9GQPY8E1ykSrbQpv4c2pUM2FQfiN1x2+p5qRVOz9rsXCYgnO40MFqwTWpZnDgcQemY8pPbsM9hlsM9hkYAqlMYOCW1wrvYcFWXE//2ZQZzfLYD+DBTR8l8zyWQa7NBt+7Ed8bzPY3xKlkIn3gsuINIMdvIYtHet3fH6kGezhNWwGJqKC3S28XcBuBkoTv7cL2M9ecP1K7cj1XteMONx4xellPztxrjMZFgtwAoS0SbQ+pSAlswxub89iHRIp522LskxWajdojLWowKl5yuvtAm7P9A6JGdkcWKReipLNz87fwG0GbLVSOzYnA49jkQZfzItf5ReptnKI7SG+xWQIF/z8usRyFapY4jI2XHGtlB0VL3rxitBG3nwphU5arlFas/hFd5gB7oSxS/XF/fRyqW9I3iAsoGIxX+9mt3+7eff+l5uDk3fMfxetv87UE1xiKgvT/vBkdEmJIneCTJJSVGkxNKhcmGV4TYhuFNmWa2LsTlNtWZquqsQuYSQ7p93RrWk1hafqMYOQazhEZY4sg6ruzGZkJT2uYp32z7zTtavjFTv4DnOcnvCxokHV2cU3M1flNZpWSYOZUze0gO1lC3CeCrq5kjc72Xq6Q0QBf6xROF75VovTXvGEFq8XrpY9QSiGJMMS/gzJX+6C6KiBu6aXCozjMfhuDocRu+kUTow5wg/in0+EKaTRlYwhn6cvXrUP0nevkfiAH+dwIBbj8uvCqLGteREycujyUYN/i/a7PmEp0JFdz+VP8LFFwjKu/xUoCY5AKTQWli47r40CjZ+xsAaELFTT1mixV9Hk59YHETn5kjr0eb+71nqa1u5HppK6Z8WFliJo0EilIRlqSAZWcyGXheqkxwqwgDezWUZo4HT1jlYNYhkXvr67UoFCmcg1GlU/YgBQqrNtNy4hjGD8jT2MpB9vHg+90OONOZCsoy9AooIk8JgCa7gUFV1kQsws7evOqK6E1HxPOxQiPybM4eC5PFUtgssGUaWfE3KC0F6VKBwWcOjPMdKVzZ17smG1qAmYGjYHB7NHO67C0sbhOFqNI8/5equVVYWq2RzYvz5+/7EHaaoRNjQND8v+DvjI684NGmHDQKHVtgdxNM9YUFUlCjEetxokSG7ygMKPPVJzqmbgwuLac8Jc1KhQDtFLM0jYI6/ZKHPSUUSCaUtC0VTxT1qb72rk794DQ0yiw++HzYd7d+yBYsDotrD5iQB3gzJgHsSzeQT+Y5rUW+iyi7DrshQ6g5qvsKZXWIyywBEYBtPojGHLHRi2Lnn+0ZZMTy/9/zmksVC6NNl4xOyRxDLs9gC8B+FPVrwTJ12tfL19xNYobbFMpMsVSQlyWcbo/rKpPzJlRCPdfCHzWm1RJ2mOsjSkTcLyz+2apaetKtTHGbwNafjWdzXP8Up/vFIpP/nmjyUcfIWBAHOCKRnwRy5qvqoRDiPm48ZBT7FRBuVguJ/U80/uX0IFIM0Np1ofOISrc2YSecvniCgzaLBZIUFhQNk1qGkO9IIyOEfCZ/iNym/iz6fO5ZdowFerPiPPIdw5rccWy9XeoonZ4KFJEHOJI1wiuS8KHh71XdMJvQ4f3GQSx9SoX27E75da9QmXP6IW1T50lfHDpVwGLy6CO/sWP6ReBmw6uiZTluY6jIM5I1ffzx7gNbDc7qwv9+NH8+2S5lVYwIlbBtlpXiJ9F0pYZ6ubNzdGrM/S54npeDTORynZqYcu2fjS4272ZYIoHcS4IdsLvB6HYRiFRaC890PPJdtBbJygrpI4DdghMPtqNGN99XCEQ+KGUZjCXQrT8M0if1PRjptF4074guG3tmekm1OCy2jRc+k0ekYZ8KIZQ4+/9jA3NVK1GPFM4TXcZi+cC1efZs5YBZ498eqV9+YLfLlGzuawhVeweUkF43ABm8PsacqhST7v1mRonFNfkmKJSc8GO7pZaRpqDOX5dd+z/0iWf1ZCes4mzQBloWjoW/jLdRnxS7VC93s2yk9HmP3B+DCydRlwSLT7GXKXxhQld7+fpvPpzeYhz5+hPAc6rv5ep7+M6JWx8BQ8DF9LrqUtMIc8ackfO17DbgHwjpBbYN7TFqpQJ6DaWSZkpc7gsF/3kGzew5/T7dF1JGw4QkandKE0Ccfq/hAinkFElfLoO3Xmoc3QnR2gTx8GdoNfh0mlYvFTChkXx5VR6tNKXnZNaxKiuJLk6bNs41exZzmfRfNZIXFw+slPimUESfQRkz6JBhYDaDqcF79j7Ds+Ky7nLD800Hd2hx8P+j6m74PztYNBVxInTB0P3tGiAi73yfgwNbox72eZ0ajyMEJWF4iRfShRWlHw2tvqPnmitB6GG+CFVsZ4iQ7Mug8QzgnBn4WSlVifzokEqtk8zA85N8tWGbFLRnc1mDmPKT4Nw9ZAQKoP227mGjYlb1wul6KwyXm6BjHeg2ESe27WdbPUKKloJze8wiVlVuINzBwSXn7BvVl8x2uDTybYMyP89cyNVBkISdFY3D3JO0zupPHkf1BLAwQUAAAACAAAACEAzgTbbtUKAADSHwAAHAAAAHNyYy9hZXJvdmlzaW9uL2V4cGVyaW1lbnQucHm9GWuP47bx+/6KKQ8HyBetzruXSxsHCnDIownQ5g5J2iA1DIFLjW12JVIlKa992/3vxZCURb/u0hSosCtL5MxwOO+hGGM/tbxp4M/v/gbOcKmkWgFXNQjddtzwuwbhq7dfvb22btcg4IY3PXdSq4IxdnW1NLqFqlr2rjdYVSDbThsHXCntPJi9uopj4aeRd0WLjtfc8WHmn1ar4blruFtq0w7vtr/rjBZo7TDiZIth3Y67dSPvhkXfcbfer7bjbXN1dVXjElBtpNGqReWyCVx/CbUUbnYFAAOm00asr/yIM7swRZfQbSsdlAkXhVijuK9077reZXO2ko7lwAxurjtuLNLLd9+8+ZotcnC4deXPpsdJYZ2RXTbxlHErsHOQvf3pG2O0yVPqX/GmwfpdePPTkzPssF7dK/2gmJ8y6Hqj4HEPx7qdW2vFZntpFmGk2qCxUqtskifAEeYAPD4cAIbl2SzykZLg4p6v0LJZwgVd3eyM2ouBiy7IY7iW2kAHUkHG+sYZ3uycFJYE6vWzf9hIwqZX1bfdjh66ndBCO60by0aiTynzfc0rvuGyIZNms6DzgoYLaceZgx2vuv4QcoWuqnEjBVaKt5hNJyCXl0kBNhbhB63wiJM91SgKjxxgnqLVemfM/BAJbebNO4Bgp8XazkCqqATZruz75P2OO7FO3onXGVhnwmvYQTJgEWsPDiV8ehvGHrS5RxNWgRKmYdSg7SMt+LffGZRxg961iMkzruXVSy6bKHaA+PXtX94G3wtOBaWnkjHTK8sm8NKzH2guI0yBW2mdzSY+VCntBsb2YjZcWoRvZYPfeFDvS9mS/dgr4I1BXu8gEJnBYyD69AWItdYWgSvt1mj8wqANXF8H8tG2hFZLuYLSB5KgokFNpXUm80Zu0Opmg9kkMaiguDL8jMNefaW/j4Neh6W/j4NBc2X4GYdJfyXdxqGovjL+jhOtrrEp2U43+uZGFZ1j45zunGzlezQle1Pz9pdkqjHTclpMpzfJEihXa/IHwXd+bvp6nFR39ph53nY+Go4jgos1lt/yxiaDNTo0rVTSOimOEDruJCqBJyIUjbZYtdpyKcpputswUiSsWcEbLKfFq3Fo2ciuMYdgndH/ROHKYIfJzniLJd0SvZIhVfr+eCtdo5092oLlG6w6NFLXZSLMlm+rGl35ahq5D6b2bBQk1NJSWPGO00ghHawMryUqB1yIvu0bn3Fn4NYIBv/Vo3VYB0MCaQG3Aa3wlCkaQ5lmjSRPstlh1kzC156wTwTkCMmkddw4rKveCQpxskXKfEt6yNjzX6+ft9fP65+ffzd7/tfZ85/+wfIAs2o9xOAqT/6eBoGiva+lyfZy9knVQ3WoaipZToLGkj2Skp7GtENlBkuRigcjHVaUpjOaLOq+7WxGCDlIVaNy5e0kB1RCE3zJere8/lOMAh8K/GMcSkAMWnRVh/y+arHVZldZxymIhRhMooMyCKRDs6yE7pVDE+e930Lpw2UWwhGFptSR93ydBEO0fUPEPZEipJYAFKzzKLAcR49YtzT2YzRfvHi8n8HGp/L7HDaUzYORFNJhSwFbLuEe/lAC83jsaXIm9AfahXeVWpooAHR8ziwKrWrLFudFBddBkP+FhgJdrxbSkNlgXd3tHMY1RgLkolFxA2DUTRbZfwkMt+Tb5DXB3ib/m5F1RqoDvEeycOYTcFx1ksNeKrNjKT1Nclg2vV0nLhPrxYAdq41Q7eP5eiMEepsOjRUHlPDZpzFmOdngyaDeoGl4N4Nlo7kvJYrbkzqEitppDLKNbKU7qjsCs4GDA19/GTiXXtaxCDmp74Xudgfvm9v01ZeRwC2o7gOlS1JjFvQ0gFF3dBmKGqYUkt4/Xg7tAQrSxTBVyxVal0ySvPeInUHaeCilbIcCSt8FFZYvsWo0r4fChNfBHE9MLtqH1ntPJDpzRo0WW4RZZ3qvAm+TRNVmmcd4CWzDG7/pwfQ/upRsqW0gNyOqcxbe2WI+80awIBf2T6GSPoIKJGrCfzRzJmu28JHHUNgJQCGVHOFBGWfTyaRl9RBz7mlxonUWRC6BR5qVX5oWrW3g6plXNgijH2pYc1U3lKSkBa72eRh41xm9la0nCU7D36X92lBNLVdKG6zB4IpWCxl7GLws+wjxm8WvcOsqSRRbvs0ynsjw4r4nlCyWvG9cOZ3AJ3AT7JHErh+8DAITY4ClhKQfDkRFFXsQ1whGl9DKSdWHan+ge6e3BOxp3OktafAQi4AEd7jSZpewHYfkKcIltRe8o9JgLOnT67DYP75kXUZp5pdhogDKI3Fcxhh2RUjDc1DSZZy7O70t7/T2MgQ3yAlifruAFyTe+asP0JPWW3FarKbXYQ9/fmSws0/KaC4rCjDkITGBrpwPdBZpnKJ1USN29JB5Pe2hhEHu8HtV4/ZcXURZMWaraOKxXwz1I6U4NVTkY9WepkYfSn9HWRxXreya377+jM1isN5zk4C2XMklWncCO7jyABD9OD1oMdppoRs2A5acyr15l5MHf43O+gYixp3D+EJechRXvvCOqJdLKSRvxvBjhTZSrZKmh1l0TqoVlRjeEdKe1ef9km75kO3L+Jv78i92jiB1X06LPx6Vm+kGB6egdaymXiKT9aH0dB0OmuLpTojwtDmaCcdHD2vd+JM44qlmSbFn9IP1TQfthZjrlaP8MV/k8X8P+gx+4aaFvgPkYh2oa0UBnAvX+6zOV7Tj3llZYyQaQrVnSRrrzXlzW8iWYrE3z0HJIQO93PCGwcuYj+bTxZwtZRNOl9gi2rD3wmUkKK0/bzkMaONpxw/afat7VYfzjgtkkx7X1wyZd6I8rJD7reaQ6Hav1hyiJhMtjtR8EkChTT0m4EM+/dhvlUkgdUkgUSiB4lmhfEgw52gfk758rEeHTmHzvpMRdEx4mmBGdLtTYm20ku8xRq3h+ljXd6QoctocFJTHmou2+Hs193/ccfS9IdVmH2jiJvACbqbT6SGB4LMDvjqcJP8ucOtoKsapIedGjfvkCS9edJPxwDmR7khubOqW7JEE+1QkcB/u7oiNiz1dFDbBHMqQTr4CdSgp2VGF9yNGauNh00EbfoKXZNXT+STLPg7lMB3R00MOacU0O1dF5XBQL81gvni6tNJppvbMh+9Hmhq+oR3KVi5PMHNgVMKwM0gFfZFqbSHb1fe+7E9SxGXomBkpyt/kcDPN4dV0ujgDH59Six0n90ds6fQz+HmNZKyq5qYG27ctNztYc1MLylP7rHwznXpze/Pui6HI0L3ZtwIJwSSPg22koAZPWYe8Br0M5wH+Ex200jbI/eHX9Q2skVOPgWP+6QwK/6UEyqM9ztl+ji1CvxiUt4QWuao2vJF1xpO0SZcfhRL4nMOXJSQSTA4VfI+fedCCaGUTf+ITBqx8j+MnkXHhIJB5TO2LOf1SC3ZacfOuTDjc72I+yyH8TXO4XSTlQkB6PT2PNv0wWmXp0+hHVrw5g2pQ8KY5RT9WQ4CjdvciKd7RYXElGm5tefhxjS4xZyF70cnPRTbvhx0OB3OCoh6qvkVDBn3G1Q8jSPIljS7yAiV2Vfd6WrW2DEpXXdFiLbnKYohPv3wcYH3++gCrQyNQUbYaMHP4/PUxtt+eP/BI1+MqC/ngGLzex5OyQRUD8mktEmbDYzJ/LgvQ98940vSR4O/N+Tcc7tF1csB31hkOiF04zwuIV/8BUEsDBBQAAAAIAAAAIQBG7kRrnAMAAMIIAAAXAAAAc3JjL2Flcm92aXNpb24vdGlsZXMucHmNVU3PozYQvudXjDiZfR1Edm9o2dvuseqh6gWhyOAJWAWb2iaBt+p/r2zzEbJv1UZKwDOPZ575TBRF36ehE7WwYEWH0KDq0eoZmORQd8yYM3swjcBHB2MWwYzDoNEYoWQSRdHpdNOqh+v1NtpR4/UKoh+UtsCkVJZZoaQ5nU4cb2As09aQDmVj2wyEtBSMeMflVd1Rd2zI4NYpZmM4f4NOGFsIacvsBAAgbhAuw9ccUlDaX98OUllI3WmxBF/hEi66j2bCIPzOuhG/a600iX5VRlhxR+CiR+kCMj7uVxug8c9RaORR7K11zFjIoWcTSenK6Oy5BICxOCyACwWtRsmJZ/oJyAXOq+04DnCNdtQSjNIWOTFoSfFJM9mgN++cvcGFeqtxEJRxvOT0ISRXD0Megm85bVE0rf3/Cbbj0OGS4oVLsaWNTBRmCr2QZII3b46C9xYH6bxJg98lKPe5KQ0zCLlWPgACp43OET49wb2XD9HlEv2gkYvaEi/sFceOhkbpWYN0k2ZgrF41jXkP+fBn1/VPx2OKgqxW8nYQCDUezhzvon520rPpyjFUAHL4kqb0FC8tHIbDKl23XuCnxx/vws+UGsyKqpitW+RX2ZuTB7cUHpCH8BLTsgGL7HMZWGp3MYfCNU1K4UGhjUs3Mi4DkOcQPVrVYQTYGdz7hjqjLgsvKa7UhIaCqZV2T78L0Dso6fI9bUVLKcwphelCYb64Cno2T7OHZuz8zLgaJYeyrR8fVDGn2XyhMKXZdCnpC6Ax77n/PSpcgXL384JXYy7UeBSGWuXhcVQtVcuX51F5R10pg/kP1pmne3GRhjT4lEG+RJr47G0KcYMKhIFflES3qDqUpIpdTdLsNRIr5IibcJqnGXKoEveScLSsbkmc1J2SSPbJcdoio1CkFD6XJbzlMKU/ay8UvgTtvGs91YQNA0pOHHQ3G4q/6qrE5XgjscOW3njCdWbn2inZrGhx80vau3zujrBzSr9Mie+d/2hCPzBJzSzxkJg+SQL6IFoubiS2kXCNz6Odyh/oN/fT4JGPOVDXXnGRLc0SmsBN1k+21iTqxi14wtHUKLmQTf6bHvHVhFaPMGTbcFVqWrxTcH/AjdJ+p76LYR8gT7JwHsukHkYSJ1a53U7ilfm/KJdwPtR668ve8n10HPPced2LqB5bB/wVrUSvgkfZRptCVFVqijIodltwDobdy5yWFCJPOMoC8b8P/5HOy6FL/gFQSwECFAAUAAAACAAAACEAeLVKHWABAABRAgAADgAAAAAAAAAAAAAAgAEAAAAAcHlwcm9qZWN0LnRvbWxQSwECFAAUAAAACAAAACEA++muwbsJAABCFAAACQAAAAAAAAAAAAAAgAGMAQAAUkVBRE1FLm1kUEsBAhQAFAAAAAgAAAAhAEwp7OcyAAAAMAAAABoAAAAAAAAAAAAAAIABbgsAAHNyYy9hZXJvdmlzaW9uL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAAAAhAKpxfQnLAgAAywgAABUAAAAAAAAAAAAAAIAB2AsAAHNyYy9hZXJvdmlzaW9uL2NsaS5weVBLAQIUABQAAAAIAAAAIQCZMW3u7QkAAFgcAAAWAAAAAAAAAAAAAACAAdYOAABzcmMvYWVyb3Zpc2lvbi9kYXRhLnB5UEsBAhQAFAAAAAgAAAAhAM4E227VCgAA0h8AABwAAAAAAAAAAAAAAIAB9xgAAHNyYy9hZXJvdmlzaW9uL2V4cGVyaW1lbnQucHlQSwECFAAUAAAACAAAACEARu5Ea5wDAADCCAAAFwAAAAAAAAAAAAAAgAEGJAAAc3JjL2Flcm92aXNpb24vdGlsZXMucHlQSwUGAAAAAAcABwDRAQAA1ycAAAAA')
assert hashlib.sha256(bundle).hexdigest() == 'd51f504e17ef3ea8ebf1afc9cd0dbb657a1fda1b994a246f3197d09ea0cf74ed'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        assert (PROJECT / member).resolve().is_relative_to(PROJECT)
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
# Save the source that produced the run alongside its outputs.
(PERSIST / 'source-d51f504e17ef.zip').write_bytes(bundle)
def run(*args):
    subprocess.run([sys.executable, '-u', '-m', 'aerovision.cli', *map(str, args)], check=True)
run('doctor')


In [ ]:
# Dataset stays on Colab scratch disk for fast image loading.
run('prepare', '--train', 800, '--val', 200)
DATA = PROJECT / 'data/mini-t800-v200-s42/dataset.yaml'
subset = DATA.parent
for filename in ['manifest.json', 'dataset.yaml']:
    (PERSIST / filename).write_bytes((subset / filename).read_bytes())
# Only small run artifacts/checkpoints go to Drive, not thousands of dataset images.
(PERSIST / 'runs').mkdir(exist_ok=True)
if not pathlib.Path('runs').exists():
    pathlib.Path('runs').symlink_to(PERSIST / 'runs', target_is_directory=True)
assert pathlib.Path('runs').resolve() == (PERSIST / 'runs').resolve()


In [ ]:
# Keep the same name to resume. Choose a NEW name for a separate experiment.
RUN_NAME = 'colab-pilot-v1'
EPOCHS = 20
run_dir = pathlib.Path('runs') / RUN_NAME
last = run_dir / 'weights/last.pt'
best = run_dir / 'weights/best.pt'
completed_epochs = 0
if (run_dir / 'results.csv').exists():
    with (run_dir / 'results.csv').open() as f:
        records = list(csv.DictReader(f))
    if records:
        completed_epochs = int(float(records[-1]['epoch']))
if completed_epochs >= EPOCHS and best.exists():
    print('Training already completed; proceeding to evaluation.')
elif last.exists():
    print('Resuming saved checkpoint; last recorded epoch:', completed_epochs)
    run('train', '--data', DATA, '--name', RUN_NAME, '--resume', last, '--device', '0')
else:
    if run_dir.exists():
        raise RuntimeError('An incomplete run folder has no checkpoint. Choose a new RUN_NAME.')
    run('train', '--data', DATA, '--epochs', EPOCHS, '--batch', 2,
        '--workers', 2, '--name', RUN_NAME, '--device', '0')


In [ ]:
# Both modes use the same checkpoint, images and prediction settings.
COMPARISON = pathlib.Path('runs') / (RUN_NAME + '-comparison')
run('compare', '--data', DATA, '--weights', best, '--output', COMPARISON, '--device', '0')
report = json.loads((COMPARISON / 'comparison.json').read_text())
for mode, metrics in report['modes'].items():
    print(mode, json.dumps(metrics, indent=2))


In [ ]:
# Compact result bundle, already stored in Drive; download a copy if desired.
export = PERSIST / (RUN_NAME + '-results.zip')
with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as archive:
    for directory in [run_dir, COMPARISON]:
        for path in directory.rglob('*'):
            if path.is_file() and (path.suffix in {'.json', '.csv', '.yaml', '.png'}
                                  or path.name in {'best.pt', 'last.pt'}):
                # Exclude dataset preview images; results.png is a learning curve.
                if path.name.startswith(('train_batch', 'val_batch')) or path.name == 'labels.jpg':
                    continue
                archive.write(path, path.relative_to('runs'))
print('Saved:', export)
from google.colab import files
files.download(str(export))


## Interpretation
These are exploratory COCO-style validation metrics, not official VisDrone scores.
Ignored regions use an explicit crowd approximation. The validation set is fixed;
test-dev has not been used. Small-object recall gains must be weighed against latency
and false positives. Record the GPU actually assigned; do not compare its latency
directly with laptop timings.

Free GPU type and session duration vary: https://research.google.com/colaboratory/faq.html
After completing the experiment, disconnect the runtime to release its resources.
